# Apri MozartAI

Scegli **Senza Drive** oppure **Usa il mio Drive**, esegui la cella e apri il link `gradio.live` nell'output. Lascia la cella in esecuzione mentre usi l'app.

- **Senza Drive** scarica il checkpoint pubblico del progetto quando è configurato. Nel frattempo, puoi caricare una copia del `best.pt` addestrato dal tuo dispositivo.
- **Usa il mio Drive** cerca il checkpoint in `MyDrive/MozartAI/checkpoints_v2/best.pt`.


In [ ]:
#@title Avvia MozartAI
origine_modello = "Senza Drive" #@param ["Senza Drive", "Usa il mio Drive"]

import importlib.util
import os
from pathlib import Path
import subprocess
import sys
import tempfile

# Il responsabile del progetto può inserire l'ID del proprio best.pt
# condiviso in sola lettura con "Chiunque abbia il link".
ID_CHECKPOINT_PUBBLICO = ""
MODELLO_DRIVE = Path("/content/drive/MyDrive/MozartAI/checkpoints_v2/best.pt")

cartella = Path(tempfile.mkdtemp(prefix="mozartai-"))
repo = cartella / "MozartAI"

if origine_modello == "Usa il mio Drive":
    from google.colab import drive

    drive.mount("/content/drive")
    modello = MODELLO_DRIVE
elif ID_CHECKPOINT_PUBBLICO:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown

    modello = cartella / "best.pt"
    if not gdown.download(id=ID_CHECKPOINT_PUBBLICO, output=str(modello), quiet=False):
        raise RuntimeError("Download di best.pt fallito. Controlla la condivisione del checkpoint.")

else:
    from google.colab import files

    print("Carica il best.pt addestrato del progetto dal tuo dispositivo.")
    caricati = files.upload()
    if "best.pt" not in caricati:
        raise FileNotFoundError("Seleziona il checkpoint del progetto con nome best.pt e riesegui la cella.")
    modello = cartella / "best.pt"
    modello.write_bytes(caricati["best.pt"])
    del caricati

if not modello.is_file() or modello.stat().st_size == 0:
    raise FileNotFoundError(f"Checkpoint best.pt mancante o vuoto: {modello}")

subprocess.run([
    "git", "clone", "--depth", "1", "--branch", "main",
    "https://github.com/PaoloniCoder/MozartAI.git", str(repo),
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "gradio==6.32.0", "pretty_midi", "pydantic>=2.12.5,<3", "starlette>=1.3.1,<2",
], check=True)

os.environ["MOZART_MODELLO"] = str(modello)
spec = importlib.util.spec_from_file_location("mozart_app", repo / "app.py")
mozart_app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mozart_app)

print("Modello:", mozart_app.precarica())
mozart_app.avvia_app(share=True, debug=True)
